In [1]:
import numpy as np
import math
from sklearn.model_selection import train_test_split
import torch.nn as nn
import matplotlib.pyplot as plt
import random
import matplotlib as mpl
import os
import gc
import pandas as pd
import csv
from numpy import *
from datetime import date
import time
import builtins
from sax import sax_tokenizer

In [2]:
seed = 50 #[42,43,50]

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
import os

# # Step 1: Load the datasets
# x_train = np.load("x_train.npy", allow_pickle=True)
# y_train = np.load("y_train.npy", allow_pickle=True)
# x_test = np.load("x_test.npy", allow_pickle=True)
# y_test = np.load("y_test.npy", allow_pickle=True)

# # Step 2: Combine them
# X = np.concatenate([x_train, x_test], axis=0)
# y = np.concatenate([y_train, y_test], axis=0)



# # Step 3: First split: Train+Valid and Test
# x_train_valid, x_test, y_train_valid, y_test = train_test_split(
#     X, y, test_size=0.11, random_state=seed, stratify=y
# )

# # Step 4: Second split: Train and Valid
# x_train, x_valid, y_train, y_valid = train_test_split(
#     x_train_valid, y_train_valid, test_size=0.11, random_state=seed, stratify=y_train_valid
# )
# # (0.25 * 0.8 = 0.2 --> 60% train, 20% valid, 20% test)

# # Step 5: Save them
# np.save("x_train.npy", x_train)
# np.save("y_train.npy", y_train)
# np.save("x_valid.npy", x_valid)
# np.save("y_valid.npy", y_valid)
# np.save("x_test.npy", x_test)
# np.save("y_test.npy", y_test)

# print(len(x_train), len(x_test), len(x_valid))


In [4]:
category = 70
word_len = 1

def convert_to_sax(input_x):
    x_sax = np.zeros(input_x.shape)
    for i in range(len(x_sax)):
        start = 0
        for j in range(input_x.shape[-1]):
            temp = sax_tokenizer(input_x[i,:,j].tolist(),alphabet_size=category, word_length=word_len) #+ start
            x_sax[i,:,j] =  np.array(temp) + start
            start += category
        if i%100 == 0:
            print(f'Done with {i}')        
    return x_sax      

In [5]:
idx = 1
x_train = np.load(f'./x_train.npy', allow_pickle=True)
x_valid = np.load(f'./x_valid.npy', allow_pickle=True)
x_test = np.load(f'./x_test.npy', allow_pickle=True)

sax_train = convert_to_sax(x_train)
sax_valid = convert_to_sax(x_valid)
sax_test = convert_to_sax(x_test)

assert x_train.shape[0]==sax_train.shape[0]
assert x_valid.shape[0]==sax_valid.shape[0]
assert x_test.shape[0]==sax_test.shape[0]

Done with 0
Done with 100
Done with 200
Done with 300
Done with 400
Done with 500
Done with 600
Done with 700
Done with 800
Done with 900
Done with 1000
Done with 0
Done with 100
Done with 0
Done with 100


In [6]:
print(sax_train.shape, sax_valid.shape, sax_test.shape)

(1050, 500, 1) (130, 500, 1) (146, 500, 1)


In [7]:
np.unique(sax_train), len(np.unique(sax_train))

(array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12.,
        13., 14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25.,
        26., 27., 28., 29., 30., 31., 32., 33., 34., 35., 36., 37., 38.,
        39., 40., 41., 42., 43., 44., 45., 46., 47., 48., 49., 50., 51.,
        52., 53., 54., 55., 56., 57., 58., 59., 60., 61., 62., 63., 64.,
        65., 66., 67., 68., 69.]),
 70)

In [8]:
def onehotencoding(x_sax):
    x_sax = x_sax.astype(int)
    x_sax_ohe = np.zeros((x_sax.shape[0], x_sax.shape[1], category*x_sax.shape[-1]))

    for i in range(len(x_sax_ohe)):
        for j in range(x_sax_ohe.shape[1]): 
            idx = x_sax[i,j,:].tolist()
            x_sax_ohe[i,j,idx] = 1
    
    return x_sax_ohe

In [9]:
sax_train_ohe = onehotencoding(sax_train)
sax_valid_ohe = onehotencoding(sax_valid)
sax_test_ohe = onehotencoding(sax_test)

In [10]:
print(sax_train[100,0:2,:], sax_train_ohe[100,1,:])
# print(sax_valid[100,0:10,:], sax_valid_ohe[100,2,:])

[[4.]
 [2.]] [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [11]:
np.save('./sax_train', sax_train_ohe)
np.save('./sax_valid', sax_valid_ohe)
np.save('./sax_test', sax_test_ohe)